# Draft Helper

Startup notebook for loading hitter and pitcher leaderboards via pybaseball's built-in cache behavior and browsing them in tabbed grids.

In [ ]:
from __future__ import annotations

from datetime import date
from typing import Dict, List

import pandas as pd
import ipywidgets as widgets
import ipydatagrid as dg
from IPython.display import display
from pybaseball import batting_stats, pitching_stats, cache

cache.enable()
print(cache.config.cache_directory)

In [ ]:
def normalize_core_columns(df: pd.DataFrame, player_type: str) -> pd.DataFrame:
    col_aliases = {
        'PlayerName': ['Name', 'name', 'player_name', 'player'],
        'PlayerID': ['IDfg', 'ID', 'playerid', 'player_id', 'mlb_id'],
        'Team': ['Team', 'Tm', 'team'],
        'Pos': ['Pos', 'position', 'Position'],
        'Season': ['Season', 'season', 'year', 'Year'],
    }

    normalized = pd.DataFrame(index=df.index)
    for canonical, aliases in col_aliases.items():
        normalized[canonical] = pd.NA
        for alias in aliases:
            if alias in df.columns:
                normalized[canonical] = df[alias]
                break

    if player_type == 'hitters':
        ranking_stats = ['HR', 'SB', 'R', 'RBI', 'AVG', 'OBP', 'SLG', 'wRC+', 'WAR']
        sort_cols = [c for c in ['WAR', 'HR', 'SB'] if c in df.columns]
    else:
        ranking_stats = ['W', 'SV', 'SO', 'K/9', 'ERA', 'WHIP', 'WAR']
        sort_cols = [c for c in ['WAR', 'SO', 'SV'] if c in df.columns]

    for col in ranking_stats:
        normalized[col] = df[col] if col in df.columns else pd.NA

    if sort_cols:
        normalized = normalized.sort_values(by=sort_cols, ascending=[False] * len(sort_cols), na_position='last')

    normalized = normalized.reset_index(drop=True)
    normalized.insert(0, 'Rank', normalized.index + 1)
    normalized.insert(0, 'Drafted', False)
    return normalized


hitters = normalize_core_columns(hitters_raw, player_type='hitters')
pitchers = normalize_core_columns(pitchers_raw, player_type='pitchers')

hitters.head(3), pitchers.head(3)


In [ ]:
show_drafted_toggle = widgets.Checkbox(
    value=False,
    description='Show drafted players',
    indent=False,
)


def filtered_view(df: pd.DataFrame) -> pd.DataFrame:
    if show_drafted_toggle.value:
        return df
    return df.loc[~df['Drafted']]


def make_grid(df: pd.DataFrame) -> dg.DataGrid:
    return dg.DataGrid(
        df,
        editable=True,
        selection_mode='row',
        base_row_size=28,
        layout=widgets.Layout(width='100%', height='650px'),
    )


def build_tab() -> widgets.Tab:
    hitter_view = filtered_view(hitters)
    pitcher_view = filtered_view(pitchers)

    hitter_grid = make_grid(hitter_view)
    pitcher_grid = make_grid(pitcher_view)

    def apply_edit(event, source_df: pd.DataFrame, visible_df: pd.DataFrame) -> None:
        row = event.get('row')
        if row is None or row >= len(visible_df):
            return

        column = event.get('column', event.get('column_name'))
        if isinstance(column, int):
            if column < 0 or column >= len(visible_df.columns):
                return
            column = visible_df.columns[column]

        if column != 'Drafted':
            return

        value = event.get('value', event.get('new', event.get('cell_value')))
        source_index = visible_df.index[row]
        source_df.at[source_index, 'Drafted'] = bool(value)
        refresh_display()

    hitter_grid.on_cell_change(lambda event: apply_edit(event, hitters, hitter_view))
    pitcher_grid.on_cell_change(lambda event: apply_edit(event, pitchers, pitcher_view))

    tab = widgets.Tab(children=[hitter_grid, pitcher_grid])
    tab.set_title(0, 'Hitters')
    tab.set_title(1, 'Pitchers')
    return tab


container = widgets.VBox()


def refresh_display(*_args) -> None:
    container.children = [show_drafted_toggle, build_tab()]


show_drafted_toggle.observe(refresh_display, names='value')
refresh_display()
display(container)
